<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
!pip install -q tensorflow-recommenders tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.4 MB/s eta 0:00:00


In [3]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [6]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"
FEATURE_PATH = "/content/drive/MyDrive/Recommendation_Engine/features"
VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

In [7]:
customers_df = spark.read.parquet(f"{PROCESSED_PATH}/customers_clean.parquet")

articles_df = spark.read.parquet(f"{PROCESSED_PATH}/articles_clean.parquet")

transactions_df = spark.read.parquet(f"{PROCESSED_PATH}/transactions_clean.parquet")

In [8]:
recency_df = spark.read.parquet(f"{FEATURE_PATH}/recency.parquet")

product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/product_popularity.parquet"
)

monthly_product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/monthly_product_popularity.parquet"
)

In [9]:
customer_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

product_type_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/product_type_vocab.parquet"
)

department_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/department_vocab.parquet"
)

color_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/color_vocab.parquet"
)

In [10]:
print("Processed Datasets")
print("------------------")
print("Customers      :", customers_df.count())
print("Articles       :", articles_df.count())
print("Transactions   :", transactions_df.count())

print("\nFeature Tables")
print("------------------")
print("Recency                :", recency_df.count())
print("Product Popularity     :", product_popularity_df.count())
print("Monthly Popularity     :", monthly_product_popularity_df.count())

print("\nVocabularies")
print("------------------")
print("Customer Vocabulary    :", customer_vocab.count())
print("Article Vocabulary     :", article_vocab.count())
print("Product Type Vocabulary:", product_type_vocab.count())
print("Department Vocabulary  :", department_vocab.count())
print("Color Vocabulary       :", color_vocab.count())

Processed Datasets
------------------
Customers      : 1371980
Articles       : 105542
Transactions   : 31788324

Feature Tables
------------------
Recency                : 1362281
Product Popularity     : 104547
Monthly Popularity     : 768883

Vocabularies
------------------
Customer Vocabulary    : 1371980
Article Vocabulary     : 105542
Product Type Vocabulary: 131
Department Vocabulary  : 250
Color Vocabulary       : 50


#Create the Interaction Dataset

In [11]:
interactions_df = transactions_df.select(
    "customer_id",
    "article_id"
)

print("Total Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Total Interactions: 31788324
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016003 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016001 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|682236013 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016016 |
|aaa7a0483dd5b9e395d95324dcbfeb617af9800f39487d4b6aaee662bcd384c7|783335003 |
|aaa7b371465a823fec4312ef0f2807f924d54e5d41afb686223b76266bd9c599|563519008 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|783056001 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|695325016 |
|aaa8f491632b9022bf20aa444c793bdf23621bd9463c050e86b73ada4cb059b6|757333001 |
|aaa8f491632b9022bf20aa444c793bdf23

#Remove Duplicate User–Item Pairs

In [12]:
interactions_df = interactions_df.dropDuplicates()

print("Unique User-Item Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Unique User-Item Interactions: 27306439
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaac535f79b71437632d6001ebd960766da3000d7c455cd7a0500dab24cfd50e|799507001 |
|abd2537848661862039af47c8bcee2f9620c5fdc3b13c1af1f66f1f485c51de7|399087021 |
|ac2a5d7aa83653f77dbb343a90ebb705fb3c0a1e2683dbf74b1e08dab042c9bd|821152001 |
|ac43e628b476ec53ee48233d5ff7ad26d91b114095b51c8bdf8f5b560d101218|835730001 |
|acd03ec982613dcc026b69a4323f1db291cfbcd6a20173fab14c1063b7c014f2|708473003 |
|acfcd9df9a2a130cc54f547ea5f5829c5ff1913c9fda8e5b29a7badcbf544e26|737222004 |
|ad410adca76cb24d968194c0c2cf02d4714c2b8a639c9377c61020dc1972e8ef|734623002 |
|adb4d1ca1ae86f0a4592ba7ee5d586662945a45bb8d5a76761d971538f2c7980|728703008 |
|adceb8b35d5250062e3bd8b2a5025ee782874c798227c143ef6691488c75fb4d|399223001 |
|adf5b91a4a8092d8f2ce64e

#Create a Training Sample

In [13]:
from pyspark.sql.functions import rand

training_sample = (
    interactions_df
    .orderBy(rand())
    .sample(withReplacement=False, fraction=0.05, seed=42)
)

print("Training Sample Size:", training_sample.count())

training_sample.show(10, truncate=False)

Training Sample Size: 1366101
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|62181c2c025b4e586502b3f1ddc15270ca89e2aee16d861bebaa01e87bc75d07|650677004 |
|c2b4ca1c4089522d058c3553d4a3451e5ac805d31ca673a1838ffd5e0e644b13|689165001 |
|8dc3d60ce8891ee70a5cf6b2504191b25d488b8ea38799e32354a913d32af714|744180001 |
|c519b11fe724c112124b5e92437bd99260e0df46de0de6ec9ec1bc0c9dd6e677|739590033 |
|a12cc5bb7fb4c320a80dcb96e2733ac535b04a28f4649ae613e7e0f060bbd32b|832458002 |
|153907e9c4a15aa00eecd9998ca2ce869b980810896203cd35a70af815151fea|671505001 |
|92b26d66df62425f3b57f7621bfd5101972309fb06797405de701dfaf0bc64ab|524825012 |
|1e0c1e57c9daa439990e17a84603224b976818c20143649bbc6baf344609ddbc|849103001 |
|28e4d794ab3e153bd2ba09061d92edfe481463c552658639019e5ea45ddd4004|768912001 |
|cadbc01685cbdee0eccf3517506fc07a9

#Convert to NumPy

In [14]:
training_pd = training_sample.toPandas()

customer_ids = training_pd["customer_id"].astype(str).values
article_ids = training_pd["article_id"].astype(str).values

print(customer_ids[:5])
print(article_ids[:5])

['62181c2c025b4e586502b3f1ddc15270ca89e2aee16d861bebaa01e87bc75d07'
 'c2b4ca1c4089522d058c3553d4a3451e5ac805d31ca673a1838ffd5e0e644b13'
 '8dc3d60ce8891ee70a5cf6b2504191b25d488b8ea38799e32354a913d32af714'
 'c519b11fe724c112124b5e92437bd99260e0df46de0de6ec9ec1bc0c9dd6e677'
 'a12cc5bb7fb4c320a80dcb96e2733ac535b04a28f4649ae613e7e0f060bbd32b']
['650677004' '689165001' '744180001' '739590033' '832458002']


#Build the TensorFlow Dataset

In [15]:
import tensorflow as tf

interactions_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": customer_ids,
    "article_id": article_ids
})

In [16]:
for sample in interactions_ds.take(5):
    print(sample)

{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'62181c2c025b4e586502b3f1ddc15270ca89e2aee16d861bebaa01e87bc75d07'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'650677004'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'c2b4ca1c4089522d058c3553d4a3451e5ac805d31ca673a1838ffd5e0e644b13'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'689165001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'8dc3d60ce8891ee70a5cf6b2504191b25d488b8ea38799e32354a913d32af714'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'744180001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'c519b11fe724c112124b5e92437bd99260e0df46de0de6ec9ec1bc0c9dd6e677'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'739590033'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'a12cc5bb7fb4c320a80dcb96e2733ac535b04a28f4649ae613e7e0f060bbd32b'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'832458002'>}


#Prepare for Training

In [17]:
BATCH_SIZE = 8192

train_ds = (
    interactions_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Create the Lookup Layers

In [18]:
customer_ids_vocab = (
    customer_vocab
    .select("customer_id")
    .toPandas()["customer_id"]
    .astype(str)
    .tolist()
)

article_ids_vocab = (
    article_vocab
    .select("article_id")
    .toPandas()["article_id"]
    .astype(str)
    .tolist()
)

print("Customers:", len(customer_ids_vocab))
print("Articles :", len(article_ids_vocab))

Customers: 1371980
Articles : 105542


#Build the Lookup Layers

In [19]:
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_ids_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_ids_vocab,
    mask_token=None
)

#Test the Lookup

In [20]:
sample_customer = customer_ids[0]
sample_article = article_ids[0]

print("Customer Index:", customer_lookup(tf.constant(sample_customer)).numpy())
print("Article Index :", article_lookup(tf.constant(sample_article)).numpy())

Customer Index: 1214262
Article Index : 36408


#Build the Query Tower

In [21]:
query_tower = tf.keras.Sequential([
    customer_lookup,

    tf.keras.layers.Embedding(
        input_dim=customer_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Query Tower

In [22]:
sample_embedding = query_tower(
    tf.constant([customer_ids[0]])
)

print(sample_embedding.shape)
print(sample_embedding.numpy())

(1, 64)
[[ 7.9709552e-03  7.7165556e-03 -1.1099819e-03  1.5449164e-02
  -1.7127613e-02  2.4365280e-03 -3.2754868e-02 -1.2573959e-02
  -1.5384950e-02 -3.6176160e-02  2.7939677e-07  4.9390066e-03
  -4.2379372e-02 -3.2361723e-02  9.8169502e-03  2.6329096e-02
  -6.2757283e-03 -2.4475401e-02  2.1827314e-03  9.6236942e-03
  -2.3940308e-03 -1.8589228e-02 -2.4116766e-02 -1.5059710e-03
   9.6892286e-03  4.2858265e-02 -1.2081578e-02 -1.0181684e-03
  -5.6905774e-03  7.6469444e-03  2.6915222e-04 -1.1949355e-02
   1.0064985e-02  1.6077455e-02 -9.1521731e-03  1.1813736e-02
   8.7664118e-03  1.1213049e-02  2.5119979e-02  1.3144056e-02
   2.3702618e-02  3.9325621e-02  2.8825991e-02 -2.2657122e-03
   1.9328604e-03  1.2467660e-02  1.3818743e-02 -9.5222723e-03
  -7.8175589e-04  3.1323489e-02  3.6667637e-04 -1.8995691e-02
  -2.0227838e-02 -3.5966579e-02  1.8003425e-02  6.6519804e-02
  -1.5049510e-02  3.9819493e-03  1.3989156e-02 -7.4794432e-03
  -7.0833871e-03 -1.1333807e-02  3.1893335e-02 -6.6019222e-04]

#Build the Candidate Tower

In [23]:
candidate_tower = tf.keras.Sequential([
    article_lookup,

    tf.keras.layers.Embedding(
        input_dim=article_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Candidate Tower

In [24]:
sample_item_embedding = candidate_tower(
    tf.constant([article_ids[0]])
)

print(sample_item_embedding.shape)
print(sample_item_embedding.numpy())

(1, 64)
[[-0.00906052  0.03612697  0.01142062 -0.01149675 -0.007494    0.01759071
   0.00961226  0.02230729 -0.01042003 -0.00931771 -0.01406377  0.02385724
   0.00212987 -0.0016662  -0.01057331 -0.00694579 -0.04519556 -0.03000993
   0.01199908 -0.01100773  0.01146008  0.01634782  0.00054378  0.01221247
  -0.01098135  0.00032291 -0.00266492  0.02368453  0.04027753  0.05795132
   0.02361972 -0.01586828 -0.01057728  0.02458864 -0.0046505  -0.01034683
   0.02906418  0.02495401  0.04242328 -0.00707592  0.00083517  0.02436683
   0.04073291 -0.00946545  0.00035415 -0.00131623 -0.0102624   0.00534587
  -0.01370452  0.00475841  0.01601475  0.00869768  0.00340117 -0.00457242
   0.00463236 -0.03918909  0.0213873   0.0102577   0.00305158  0.00999706
  -0.0031345  -0.01660357  0.02281738 -0.01573521]]


#Verify Both Towers

In [25]:
print("User Embedding Shape :", sample_embedding.shape)
print("Item Embedding Shape :", sample_item_embedding.shape)

User Embedding Shape : (1, 64)
Item Embedding Shape : (1, 64)


#Create the Candidate Dataset

In [26]:
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(article_ids_vocab)
    .batch(1024)
)

In [27]:
for batch in candidate_dataset.take(1):
    print(batch[:5])

tf.Tensor([b'108775015' b'108775044' b'108775051' b'110065001' b'110065002'], shape=(5,), dtype=string)


#Build the Retrieval Task

In [28]:
import tensorflow_recommenders as tfrs

retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=candidate_dataset.map(candidate_tower)
    )
)

#Build the Complete Recommendation Model

In [29]:
class HMRecommendationModel(tfrs.models.Model):

    def __init__(self, query_model, candidate_model):
        super().__init__()

        self.query_model = query_model
        self.candidate_model = candidate_model

        self.task = retrieval_task

    def compute_loss(self, features, training=False):

        user_embeddings = self.query_model(features["customer_id"])

        item_embeddings = self.candidate_model(features["article_id"])

        return self.task(
            user_embeddings,
            item_embeddings
        )

In [30]:
import os

MODEL_PATH = "/content/drive/MyDrive/Recommendation_Engine/models"

os.makedirs(MODEL_PATH, exist_ok=True)

print("Model folder created.")

Model folder created.


In [31]:
query_tower.save(f"{MODEL_PATH}/query_tower.keras")
candidate_tower.save(f"{MODEL_PATH}/candidate_tower.keras")

print("Model architectures saved.")

Model architectures saved.


In [32]:
import json

config = {
    "embedding_dimension": 64,
    "hidden_layer": 128,
    "batch_size": 8192,
    "sample_fraction": 0.05,
    "tensorflow_version": "2.20.0",
    "tfrs_version": "0.7.7"
}

with open(f"{MODEL_PATH}/model_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved.")

Configuration saved.


Instantiate the Model

In [33]:
model = HMRecommendationModel(
    query_model=query_tower,
    candidate_model=candidate_tower
)

print(model)

#Prepare the Dataset

In [34]:
dataset_size = len(customer_ids)

train_size = int(0.8 * dataset_size)

train_ds = interactions_ds.take(train_size)
test_ds = interactions_ds.skip(train_size)

#Batch the Data

In [35]:
BATCH_SIZE = 8192

train_ds = (
    train_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Verify the Split

In [36]:
print("Dataset Size :", dataset_size)
print("Training Size:", train_size)
print("Testing Size :", dataset_size - train_size)

Dataset Size : 1364034
Training Size: 1091227
Testing Size : 272807


#Compile the Model

In [37]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

#Train the Model

In [38]:
import os

In [39]:
MODEL_DIR = "/content/drive/MyDrive/Recommendation_Engine/models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [40]:
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, "model_epoch_{epoch:02d}.keras"),
    save_freq="epoch",
    save_best_only=False,
    save_weights_only=False,
    verbose=1
)

In [ ]:
history = model.fit(
    train_ds,
    epochs=5,
    callbacks=[checkpoint_callback],
    verbose=1
)

Epoch 1/5
134/134 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0062 - factorized_top_k/top_5_categorical_accuracy: 0.0080 - factorized_top_k/top_10_categorical_accuracy: 0.0091 - factorized_top_k/top_50_categorical_accuracy: 0.0147 - factorized_top_k/top_100_categorical_accuracy: 0.0186 - loss: 73645.9149 - regularization_loss: 0.0000e+00 - total_loss: 73645.9149 
Epoch 1: saving model to /content/drive/MyDrive/Recommendation_Engine/models/model_epoch_01.keras


/usr/local/lib/python3.12/dist-packages/tf_keras/src/saving/saving_api.py:227: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  saving_lib.save_model(model, local_filepath)


134/134 [==============================] - 4650s 34s/step - factorized_top_k/top_1_categorical_accuracy: 0.0062 - factorized_top_k/top_5_categorical_accuracy: 0.0080 - factorized_top_k/top_10_categorical_accuracy: 0.0091 - factorized_top_k/top_50_categorical_accuracy: 0.0147 - factorized_top_k/top_100_categorical_accuracy: 0.0186 - loss: 73193.4835 - regularization_loss: 0.0000e+00 - total_loss: 73193.4835
Epoch 2/5
134/134 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0026 - factorized_top_k/top_5_categorical_accuracy: 0.0051 - factorized_top_k/top_10_categorical_accuracy: 0.0068 - factorized_top_k/top_50_categorical_accuracy: 0.0150 - factorized_top_k/top_100_categorical_accuracy: 0.0219 - loss: 72764.5631 - regularization_loss: 0.0000e+00 - total_loss: 72764.5631 
Epoch 2: saving model to /content/drive/MyDrive/Recommendation_Engine/models/model_epoch_02.keras


134/134 [==============================] - 4553s 34s/step - factorized_top_k/top_1_categorical_accuracy: 0.0026 - factorized_top_k/top_5_categorical_accuracy: 0.0051 - factorized_top_k/top_10_categorical_accuracy: 0.0068 - factorized_top_k/top_50_categorical_accuracy: 0.0150 - factorized_top_k/top_100_categorical_accuracy: 0.0219 - loss: 72317.1518 - regularization_loss: 0.0000e+00 - total_loss: 72317.1518
Epoch 3/5
 98/134 [====================>.........] - ETA: 20:24 - factorized_top_k/top_1_categorical_accuracy: 0.0096 - factorized_top_k/top_5_categorical_accuracy: 0.0176 - factorized_top_k/top_10_categorical_accuracy: 0.0228 - factorized_top_k/top_50_categorical_accuracy: 0.0432 - factorized_top_k/top_100_categorical_accuracy: 0.0588 - loss: 70110.7170 - regularization_loss: 0.0000e+00 - total_loss: 70110.7170